# Qwen Model - Metrics Evaluation

This notebook evaluates all metrics for the Qwen model using functions from `metrics/`.

**Metrics:**
1. Visual Fidelity (CLIP + IOU)
2. Code Correctness (Render Success, Errors)
3. Structural Alignment (Semantic HTML, Accessibility, Tree Edit Similarity)
4. Robustness (Degradation Rate)
5. Efficiency (Tokens/sec, Latency, VRAM)

In [9]:
# Setup and Imports
import sys
import os
import json
import asyncio
from pathlib import Path
from tqdm import tqdm
import numpy as np
import pandas as pd
from PIL import Image
from bs4 import BeautifulSoup

# Add metrics to path
sys.path.insert(0, 'metrics')

# Configuration
MODEL_NAME = "Qwen"
PREDICTIONS_DIR = Path("results_Qwen/predictions")
RENDERED_DIR = Path("results_Qwen/rendered_imgs")
REFERENCE_DIR = Path("results_Qwen/reference_imgs")
RESULTS_DIR = Path("results_Qwen/results")
ROBUSTNESS_DIR = Path("robustness_results/qwen_0.05")  # Use 0.05 strength results
SAMPLE_RANGE = range(0, 484)

print(f"Model: {MODEL_NAME}")
print(f"Predictions: {PREDICTIONS_DIR}")
print(f"Rendered: {RENDERED_DIR}")
print(f"Reference: {REFERENCE_DIR}")
print(f"Robustness: {ROBUSTNESS_DIR}")

Model: Qwen
Predictions: results_Qwen/predictions
Rendered: results_Qwen/rendered_imgs
Reference: results_Qwen/reference_imgs
Robustness: robustness_results/qwen_0.05


## Helper: DOMNode Wrapper
Required for structural alignment metrics to work with BeautifulSoup

In [2]:
class DOMNode:
    """Wrapper around BeautifulSoup to match structural_alignment interface"""
    def __init__(self, soup_element):
        self.element = soup_element
        self.tag = getattr(soup_element, 'name', "") or ""
        self.attrs = dict(soup_element.attrs) if hasattr(soup_element, 'attrs') else {}
        self.children = [DOMNode(child) for child in soup_element.children
                        if hasattr(child, 'name') and child.name] if hasattr(soup_element, 'children') else []

def parse_html_to_dom(html_string):
    """Parse HTML string to DOM tree wrapped in DOMNode"""
    try:
        soup = BeautifulSoup(html_string, 'lxml')
        body = soup.find('body')
        return DOMNode(body) if body else DOMNode(soup)
    except Exception:
        return None

def compute_stats(values):
    """Compute statistics for a list of values"""
    values = [v for v in values if v is not None]
    if not values:
        return {"mean": None, "std": None, "min": None, "max": None, "count": 0}
    return {
        "mean": float(np.mean(values)),
        "std": float(np.std(values)),
        "min": float(np.min(values)),
        "max": float(np.max(values)),
        "count": len(values)
    }

print("Helper functions loaded.")

Helper functions loaded.


---
## 1. Visual Fidelity

### 1.1 CLIP Score (metrics/CLIP.py)

In [3]:
from CLIP import calculate_clip_score

clip_scores = []

for idx in tqdm(SAMPLE_RANGE, desc="CLIP Score"):
    ref_path = REFERENCE_DIR / f"{idx}.png"
    pred_path = RENDERED_DIR / f"{idx}.png"
    
    if ref_path.exists() and pred_path.exists():
        try:
            img_ref = Image.open(ref_path)
            img_pred = Image.open(pred_path)
            score = calculate_clip_score(img_ref, img_pred)
            clip_scores.append(score)
        except Exception as e:
            clip_scores.append(None)
    else:
        clip_scores.append(None)

clip_stats = compute_stats(clip_scores)
print(f"\nCLIP Score Statistics:")
print(f"  Mean:  {clip_stats['mean']:.4f}")
print(f"  Std:   {clip_stats['std']:.4f}")
print(f"  Min:   {clip_stats['min']:.4f}")
print(f"  Max:   {clip_stats['max']:.4f}")
print(f"  Count: {clip_stats['count']}")

/root/anaconda3/envs/new_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


CLIP model loaded successfully (openai/clip-vit-base-patch32), device: cuda


CLIP Score: 100%|██████████████████████████████████████████████| 484/484 [00:21<00:00, 22.05it/s]


CLIP Score Statistics:
  Mean:  0.6754
  Std:   0.1536
  Min:   0.2137
  Max:   1.0000
  Count: 484


### 1.2 IoU Score (metrics/IOU.py)

In [4]:
from IOU import LayoutBenchmark
from datasets import load_dataset

async def calculate_iou_scores():
    """Calculate IoU scores using ground truth HTML from dataset"""
    # Load ground truth HTML from dataset
    print("Loading dataset for ground truth HTML...")
    dataset = load_dataset("SALT-NLP/Design2Code-hf", split="train")
    print(f"Dataset loaded: {len(dataset)} samples")
    
    benchmark = LayoutBenchmark()
    await benchmark.start()
    
    iou_scores = []
    
    for idx in tqdm(SAMPLE_RANGE, desc="IoU Score"):
        pred_path = PREDICTIONS_DIR / f"{idx}.html"
        
        if not pred_path.exists():
            iou_scores.append(None)
            continue
            
        try:
            # Load prediction HTML
            with open(pred_path, 'r', encoding='utf-8') as f:
                pred_html = f.read()
            
            # Load ground truth HTML from dataset
            gt_html = dataset[idx]['text']
            
            # Extract bounding boxes from both
            gt_boxes = await benchmark.get_element_bboxes(gt_html, idx, "GT")
            pred_boxes = await benchmark.get_element_bboxes(pred_html, idx, "PRED")
            
            # Calculate IoU score
            iou_score = benchmark.compare_layouts(gt_boxes, pred_boxes)
            iou_scores.append(iou_score)
            
        except Exception as e:
            iou_scores.append(None)
    
    await benchmark.stop()
    return iou_scores

# Run IoU calculation
iou_scores = await calculate_iou_scores()
iou_stats = compute_stats(iou_scores)

print(f"\nIoU Score Statistics:")
print(f"  Mean:  {iou_stats['mean']:.4f}" if iou_stats['mean'] else "  Mean:  N/A")
print(f"  Std:   {iou_stats['std']:.4f}" if iou_stats['std'] else "  Std:   N/A")
print(f"  Min:   {iou_stats['min']:.4f}" if iou_stats['min'] else "  Min:   N/A")
print(f"  Max:   {iou_stats['max']:.4f}" if iou_stats['max'] else "  Max:   N/A")
print(f"  Count: {iou_stats['count']}")

Loading dataset for ground truth HTML...
Dataset loaded: 484 samples


IoU Score: 100%|███████████████████████████████████████████████| 484/484 [04:15<00:00,  1.90it/s]


IoU Score Statistics:
  Mean:  0.1280
  Std:   0.0866
  Min:   N/A
  Max:   0.5679
  Count: 484


---
## 2. Code Correctness (metrics/correctness.py)

In [5]:
from playwright.async_api import async_playwright

async def check_correctness():
    """Check HTML files for render success and errors"""
    results = []
    
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        
        for idx in tqdm(SAMPLE_RANGE, desc="Correctness"):
            html_path = PREDICTIONS_DIR / f"{idx}.html"
            
            if not html_path.exists():
                results.append({
                    "idx": idx,
                    "render_success": False,
                    "error_count": 1,
                    "critical_error": True
                })
                continue
            
            context = await browser.new_context()
            page = await context.new_page()
            
            errors = []
            render_success = False
            
            try:
                page.on('console', lambda msg: errors.append(msg.text) if msg.type == 'error' else None)
                page.on('pageerror', lambda err: errors.append(str(err)))
                
                file_url = f'file://{html_path.resolve()}'
                await page.goto(file_url, wait_until='load', timeout=10000)
                
                # Check if page has visible content
                is_empty = await page.evaluate("""
                    () => {
                        const body = document.body;
                        if (!body) return true;
                        if (body.innerText.trim().length > 0) return false;
                        const elements = body.querySelectorAll('*');
                        for (const el of elements) {
                            if (['SCRIPT', 'STYLE', 'META', 'LINK', 'HEAD', 'TITLE'].includes(el.tagName)) continue;
                            const rect = el.getBoundingClientRect();
                            if (rect.width > 0 && rect.height > 0) {
                                const style = window.getComputedStyle(el);
                                if (style.display !== 'none' && style.visibility !== 'hidden') return false;
                            }
                        }
                        return true;
                    }
                """)
                render_success = not is_empty
                
            except Exception as e:
                errors.append(str(e))
            finally:
                await context.close()
            
            results.append({
                "idx": idx,
                "render_success": render_success,
                "error_count": len(errors),
                "critical_error": not render_success
            })
        
        await browser.close()
    
    return results

# Run correctness check
correctness_results = await check_correctness()

# Calculate statistics
render_success_count = sum(1 for r in correctness_results if r['render_success'])
total_errors = sum(r['error_count'] for r in correctness_results)
critical_errors = sum(1 for r in correctness_results if r['critical_error'])

print(f"\nCode Correctness Statistics:")
print(f"  Render Success Rate: {render_success_count}/{len(correctness_results)} ({100*render_success_count/len(correctness_results):.2f}%)")
print(f"  Total Errors: {total_errors}")
print(f"  Critical Errors: {critical_errors}")

Correctness: 100%|█████████████████████████████████████████████| 484/484 [01:07<00:00,  7.15it/s]



Code Correctness Statistics:
  Render Success Rate: 469/484 (96.90%)
  Total Errors: 5
  Critical Errors: 15


---
## 3. Structural Alignment (metrics/structural_alignment.py)

In [6]:
from structural_alignment import semantic_html_usage, accessibility_score, tree_edit_similarity
from datasets import load_dataset

# Load ground truth HTML for tree edit similarity
print("Loading dataset for ground truth HTML...")
dataset = load_dataset("SALT-NLP/Design2Code-hf", split="train")
print(f"Dataset loaded: {len(dataset)} samples")

def parse_html_to_soup(html_string):
    """Parse HTML string to BeautifulSoup element (for tree_edit_similarity)"""
    try:
        soup = BeautifulSoup(html_string, 'lxml')
        body = soup.find('body')
        return body if body else soup
    except Exception:
        return None

semantic_scores = []
accessibility_scores = []
tree_edit_scores = []

for idx in tqdm(SAMPLE_RANGE, desc="Structural Alignment"):
    html_path = PREDICTIONS_DIR / f"{idx}.html"
    
    if html_path.exists():
        try:
            with open(html_path, 'r', encoding='utf-8') as f:
                html_content = f.read()
            
            # Parse for semantic/accessibility (uses DOMNode wrapper)
            dom = parse_html_to_dom(html_content)
            # Parse for tree_edit_similarity (uses raw BeautifulSoup)
            pred_soup = parse_html_to_soup(html_content)
            gt_soup = parse_html_to_soup(dataset[idx]['text'])
            
            if dom:
                sem = semantic_html_usage(dom)
                acc = accessibility_score(dom)
                semantic_scores.append(sem)
                accessibility_scores.append(acc)
            else:
                semantic_scores.append(None)
                accessibility_scores.append(None)
            
            # Tree edit similarity (compares prediction vs ground truth)
            if pred_soup and gt_soup:
                tree_sim = tree_edit_similarity(gt_soup, pred_soup)
                tree_edit_scores.append(tree_sim)
            else:
                tree_edit_scores.append(None)
                
        except Exception as e:
            semantic_scores.append(None)
            accessibility_scores.append(None)
            tree_edit_scores.append(None)
    else:
        semantic_scores.append(None)
        accessibility_scores.append(None)
        tree_edit_scores.append(None)

semantic_stats = compute_stats(semantic_scores)
accessibility_stats = compute_stats(accessibility_scores)
tree_edit_stats = compute_stats(tree_edit_scores)

print(f"\nSemantic HTML Usage:")
print(f"  Mean:  {semantic_stats['mean']:.4f}")
print(f"  Std:   {semantic_stats['std']:.4f}")
print(f"  Count: {semantic_stats['count']}")

print(f"\nAccessibility Score:")
print(f"  Mean:  {accessibility_stats['mean']:.4f}")
print(f"  Std:   {accessibility_stats['std']:.4f}")
print(f"  Count: {accessibility_stats['count']}")

print(f"\nTree Edit Similarity:")
print(f"  Mean:  {tree_edit_stats['mean']:.4f}" if tree_edit_stats['mean'] else "  Mean:  N/A")
print(f"  Std:   {tree_edit_stats['std']:.4f}" if tree_edit_stats['std'] else "  Std:   N/A")
print(f"  Count: {tree_edit_stats['count']}")

# Combined structural alignment score
combined_structural = []
for sem, acc, tree in zip(semantic_scores, accessibility_scores, tree_edit_scores):
    if sem is not None and acc is not None and tree is not None:
        combined_structural.append((sem + acc + tree) / 3.0)
    elif sem is not None and acc is not None:
        combined_structural.append((sem + acc) / 2.0)
    else:
        combined_structural.append(None)

combined_stats = compute_stats(combined_structural)
print(f"\nCombined Structural Alignment (Semantic + Accessibility + Tree Edit) / 3:")
print(f"  Mean:  {combined_stats['mean']:.4f}" if combined_stats['mean'] else "  Mean:  N/A")
print(f"  Std:   {combined_stats['std']:.4f}" if combined_stats['std'] else "  Std:   N/A")
print(f"  Count: {combined_stats['count']}")

Loading dataset for ground truth HTML...
Dataset loaded: 484 samples


Structural Alignment: 100%|████████████████████████████████████| 484/484 [04:54<00:00,  1.65it/s]


Semantic HTML Usage:
  Mean:  0.2713
  Std:   0.2914
  Count: 484

Accessibility Score:
  Mean:  0.0062
  Std:   0.0553
  Count: 484

Tree Edit Similarity:
  Mean:  0.1809
  Std:   0.1531
  Count: 484

Combined Structural Alignment (Semantic + Accessibility + Tree Edit) / 3:
  Mean:  0.1528
  Std:   0.1248
  Count: 484


---
## 4. Robustness (metrics/robustness.py + metrics/perturb_image.py)

In [10]:
from robustness import compute_robustness_metrics
from perturb_image import perturb_image

# Check available robustness results
robustness_dirs = [
    ("qwen_0.05", Path("robustness_results/qwen_0.05")),
    ("qwen_0.3", Path("robustness_results/qwen_0.3")),
    ("qwen_0.1", Path("robustness_results/qwen_0.1")),
    ("qwen", Path("robustness_results/qwen")),
]

print("Available Robustness Results:")
for name, path in robustness_dirs:
    report_path = path / "robustness_report.json"
    if report_path.exists():
        with open(report_path, 'r') as f:
            data = json.load(f)
        strength = data.get('strength', 'unknown')
        samples = data.get('samples', 'unknown')
        print(f"  - {name}: strength={strength}, samples={samples}")
    else:
        print(f"  - {name}: Not found")

Available Robustness Results:
  - qwen_0.05: strength=0.05, samples=50
  - qwen_0.3: strength=0.3, samples=50
  - qwen_0.1: strength=unknown, samples=50
  - qwen: strength=unknown, samples=50


In [11]:
# Load the best robustness results (qwen_0.3 has both visual fidelity and structural alignment)
robustness_report_path = ROBUSTNESS_DIR / "robustness_report.json"

if robustness_report_path.exists():
    with open(robustness_report_path, 'r') as f:
        robustness_data = json.load(f)
    
    print(f"Robustness Results ({ROBUSTNESS_DIR.name}):")
    print(f"  Model: {robustness_data.get('model')}")
    print(f"  Strength: {robustness_data.get('strength')}")
    print(f"  Samples: {robustness_data.get('samples')}")
    
    # Check format (old vs new)
    deg = robustness_data.get('degradation_rate')
    if isinstance(deg, dict):
        print(f"\n  Visual Fidelity:")
        vf = robustness_data.get('visual_fidelity', {})
        print(f"    Clean Mean: {vf.get('clean', {}).get('mean', 'N/A'):.4f}")
        print(f"    Perturbed Mean: {vf.get('perturbed', {}).get('mean', 'N/A'):.4f}")
        print(f"    Drop: {deg['visual_fidelity_drop']*100:.2f}%")
        
        print(f"\n  Structural Alignment:")
        sa = robustness_data.get('structural_alignment', {})
        print(f"    Clean Mean: {sa.get('clean', {}).get('mean', 'N/A')}")
        print(f"    Perturbed Mean: {sa.get('perturbed', {}).get('mean', 'N/A')}")
        print(f"    Drop: {deg['structural_alignment_drop']*100:.2f}%")
        
        print(f"\n  Average Degradation: {deg['average']*100:.2f}%")
    elif deg is not None:
        print(f"\n  Degradation Rate (CLIP only): {deg*100:.2f}%")
        print(f"  Clean CLIP Mean: {robustness_data['clean_clip']['mean']:.4f}")
        print(f"  Perturbed CLIP Mean: {robustness_data['perturbed_clip']['mean']:.4f}")
    else:
        print("  Degradation rate not computed (missing data)")
else:
    print("No robustness results found. Run test_robustness.py first:")
    print("  python test_robustness.py --model qwen --strength 0.3 --samples 50")

Robustness Results (qwen_0.05):
  Model: qwen
  Strength: 0.05
  Samples: 50

  Visual Fidelity:
    Clean Mean: 0.6734
    Perturbed Mean: 0.7522
    Drop: 0.00%

  Structural Alignment:
    Clean Mean: 0.1912158083537693
    Perturbed Mean: 0.14352410062922077
    Drop: 24.94%

  Average Degradation: 12.47%


### 4.1 Compute Fresh Robustness (Optional)
Run this cell if you want to recalculate robustness using existing perturbed data

In [14]:
# Optional: Recalculate robustness from existing perturbed images/predictions
RECALCULATE_ROBUSTNESS = False  # Set to True to recalculate

if RECALCULATE_ROBUSTNESS and ROBUSTNESS_DIR.exists():
    clean_clip_scores = []
    perturbed_clip_scores = []
    
    perturbed_rendered_dir = ROBUSTNESS_DIR / "perturbed_rendered"
    reference_images_dir = ROBUSTNESS_DIR / "reference_images"
    
    sample_indices = list(range(50))  # First 50 samples
    
    for idx in tqdm(sample_indices, desc="Calculating Robustness"):
        ref_path = reference_images_dir / f"{idx}.png"
        clean_path = RENDERED_DIR / f"{idx}.png"
        perturbed_path = perturbed_rendered_dir / f"{idx}.png"
        
        if ref_path.exists():
            img_ref = Image.open(ref_path)
            
            # Clean CLIP
            if clean_path.exists():
                img_clean = Image.open(clean_path)
                clean_clip_scores.append(calculate_clip_score(img_ref, img_clean))
            else:
                clean_clip_scores.append(None)
            
            # Perturbed CLIP
            if perturbed_path.exists():
                img_perturbed = Image.open(perturbed_path)
                perturbed_clip_scores.append(calculate_clip_score(img_ref, img_perturbed))
            else:
                perturbed_clip_scores.append(None)
    
    # Use metrics/robustness.py
    clean_valid = [s for s in clean_clip_scores if s is not None]
    perturbed_valid = [s for s in perturbed_clip_scores if s is not None]
    
    if clean_valid and perturbed_valid:
        robustness_result = compute_robustness_metrics(clean_valid, perturbed_valid)
        print(f"\nRobustness Metrics (Recalculated):")
        print(f"  Clean CLIP Mean: {np.mean(clean_valid):.4f}")
        print(f"  Perturbed CLIP Mean: {np.mean(perturbed_valid):.4f}")
        print(f"  Degradation Rate: {robustness_result.degradation_rate*100:.2f}%")

---
## 5. Efficiency (metrics/efficiency.py)

In [7]:
from efficiency import compute_efficiency_local

# Load efficiency data from saved results
efficiency_file = RESULTS_DIR / "all_results.xlsx"

if efficiency_file.exists():
    df = pd.read_excel(efficiency_file)
    
    tokens_per_sec = df['tokens_per_second'].dropna().tolist() if 'tokens_per_second' in df.columns else []
    latency = df['latency'].dropna().tolist() if 'latency' in df.columns else []
    peak_vram = df['peak_vram_gb'].dropna().tolist() if 'peak_vram_gb' in df.columns else []
    
    print("Efficiency Metrics (from saved results):")
    
    if tokens_per_sec:
        stats = compute_stats(tokens_per_sec)
        print(f"\nTokens/Second:")
        print(f"  Mean:  {stats['mean']:.2f} tok/s")
        print(f"  Std:   {stats['std']:.2f}")
        print(f"  Min:   {stats['min']:.2f}")
        print(f"  Max:   {stats['max']:.2f}")
        print(f"  Count: {stats['count']}")
    
    if latency:
        stats = compute_stats(latency)
        print(f"\nLatency:")
        print(f"  Mean:  {stats['mean']:.2f} seconds")
        print(f"  Std:   {stats['std']:.2f}")
        print(f"  Min:   {stats['min']:.2f}")
        print(f"  Max:   {stats['max']:.2f}")
        print(f"  Count: {stats['count']}")
    
    if peak_vram:
        stats = compute_stats(peak_vram)
        print(f"\nPeak VRAM:")
        print(f"  Mean:  {stats['mean']:.2f} GB")
        print(f"  Std:   {stats['std']:.2f}")
        print(f"  Min:   {stats['min']:.2f}")
        print(f"  Max:   {stats['max']:.2f}")
        print(f"  Count: {stats['count']}")
else:
    print(f"Efficiency data not found at {efficiency_file}")

Efficiency Metrics (from saved results):

Tokens/Second:
  Mean:  47.65 tok/s
  Std:   1.12
  Min:   44.74
  Max:   50.46
  Count: 484

Latency:
  Mean:  153.95 seconds
  Std:   40.24
  Min:   38.84
  Max:   201.17
  Count: 484

Peak VRAM:
  Mean:  17.70 GB
  Std:   0.32
  Min:   16.80
  Max:   18.30
  Count: 484


---
## Summary Report

In [ ]:
print("=" * 60)
print(f"QWEN MODEL - METRICS SUMMARY")
print("=" * 60)

print(f"\n1. VISUAL FIDELITY")
print(f"   CLIP Score Mean: {clip_stats['mean']:.4f}" if clip_stats['mean'] else "   CLIP Score: Not computed")
print(f"   IoU Score Mean: {iou_stats['mean']:.4f}" if iou_stats['mean'] else "   IoU Score: Not computed")

print(f"\n2. CODE CORRECTNESS")
print(f"   Render Success Rate: {100*render_success_count/len(correctness_results):.2f}%")
print(f"   Total Errors: {total_errors}")

print(f"\n3. STRUCTURAL ALIGNMENT")
print(f"   Semantic HTML Ratio: {semantic_stats['mean']:.4f}" if semantic_stats['mean'] else "   Semantic HTML: Not computed")
print(f"   Accessibility Score: {accessibility_stats['mean']:.4f}" if accessibility_stats['mean'] else "   Accessibility: Not computed")
print(f"   Tree Edit Similarity: {tree_edit_stats['mean']:.4f}" if tree_edit_stats['mean'] else "   Tree Edit Sim: Not computed")
print(f"   Combined Score: {combined_stats['mean']:.4f}" if combined_stats['mean'] else "   Combined: Not computed")

print(f"\n4. ROBUSTNESS")
if robustness_report_path.exists():
    deg = robustness_data.get('degradation_rate')
    if isinstance(deg, dict):
        print(f"   Visual Fidelity Drop: {deg['visual_fidelity_drop']*100:.2f}%")
        print(f"   Structural Alignment Drop: {deg['structural_alignment_drop']*100:.2f}%")
        print(f"   Average Degradation: {deg['average']*100:.2f}%")
    elif deg is not None:
        print(f"   Degradation Rate: {deg*100:.2f}%")
    else:
        print("   Not computed")
else:
    print("   Not computed")

print(f"\n5. EFFICIENCY")
if efficiency_file.exists() and tokens_per_sec:
    print(f"   Tokens/Second: {np.mean(tokens_per_sec):.2f}")
    print(f"   Latency: {np.mean(latency):.2f} seconds")
    print(f"   Peak VRAM: {np.mean(peak_vram):.2f} GB")
else:
    print("   Not available")

print("\n" + "=" * 60)

---
## Save Results to JSON

In [ ]:
# Compile all results into a single JSON
from datetime import datetime

results = {
    "model": MODEL_NAME,
    "generated": datetime.now().isoformat(),
    "sample_count": len(SAMPLE_RANGE),
    "metrics": {
        "visual_fidelity": {
            "clip_score": clip_stats
        },
        "code_correctness": {
            "render_success_rate": render_success_count / len(correctness_results),
            "total_errors": total_errors,
            "critical_errors": critical_errors
        },
        "structural_alignment": {
            "semantic_html_ratio": semantic_stats,
            "accessibility_score": accessibility_stats
        },
        "robustness": robustness_data.get('degradation_rate') if robustness_report_path.exists() else None,
        "efficiency": {
            "tokens_per_second": compute_stats(tokens_per_sec) if tokens_per_sec else None,
            "latency": compute_stats(latency) if latency else None,
            "peak_vram_gb": compute_stats(peak_vram) if peak_vram else None
        }
    }
}

output_path = RESULTS_DIR / "qwen_all_metrics.json"
with open(output_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f"Results saved to: {output_path}")